# 03 - ML Training and Preprocessing Artifacts

Notebook này huấn luyện mô hình baseline supervised và đóng gói artifacts để RL warmstart dùng lại.

Đầu ra mục tiêu:
- best_traffic_model_baseline.pt
- preprocessing_artifacts.pkl
- ml_metrics.json


In [ ]:
from pathlib import Path
import os
import json
import subprocess

ROOT = Path('/workspace/ai-core')
RUN_SCRIPT = ROOT / 'scripts' / 'run_ml_train.py'
ARTIFACT_ROOT = Path(os.getenv('ML_ARTIFACT_ROOT', ROOT / 'artifacts' / 'ml'))
print('Run script:', RUN_SCRIPT)
print('Artifact root:', ARTIFACT_ROOT)


## Cấu hình chạy

Nếu muốn chạy thật trong notebook, bật RUN_TRAIN=True ở cell kế tiếp.
Mặc định cell sẽ ở chế độ dry-run để tránh chạy tốn tài nguyên ngoài ý muốn.


In [ ]:
RUN_TRAIN = False
CMD = ['python', str(RUN_SCRIPT)]
print("Command:", " ".join(CMD))
if RUN_TRAIN:
    proc = subprocess.run(CMD, cwd=str(ROOT), check=False, capture_output=True, text=True)
    print(proc.stdout[-5000:])
    print(proc.stderr[-2000:])
    print("Exit code:", proc.returncode)
else:
    print("Dry-run: chưa thực thi train.")


In [ ]:
# Kiểm tra artifact sau khi train
checkpoints = sorted((ARTIFACT_ROOT / "checkpoints").glob("*.pt")) if (ARTIFACT_ROOT / "checkpoints").exists() else []
preps = sorted((ARTIFACT_ROOT / "preprocessing").glob("*.pkl")) if (ARTIFACT_ROOT / "preprocessing").exists() else []
metrics = sorted((ARTIFACT_ROOT / "metrics").glob("*.json")) if (ARTIFACT_ROOT / "metrics").exists() else []
print("Checkpoints:", [p.name for p in checkpoints])
print("Preprocessing:", [p.name for p in preps])
print("Metrics:", [p.name for p in metrics])


In [ ]:
# Tạo symlink/tên chuẩn để warmstart dễ dùng (tuỳ chọn)
baseline_ckpt = checkpoints[0] if checkpoints else None
baseline_prep = preps[0] if preps else None
if baseline_ckpt:
    target = ARTIFACT_ROOT / "checkpoints" / "best_traffic_model_baseline.pt"
    if target.exists() or target.is_symlink():
        target.unlink()
    target.symlink_to(baseline_ckpt.name)
    print("Linked:", target, "->", baseline_ckpt.name)
if baseline_prep:
    target = ARTIFACT_ROOT / "preprocessing" / "preprocessing_artifacts.pkl"
    if target.exists() or target.is_symlink():
        target.unlink()
    target.symlink_to(baseline_prep.name)
    print("Linked:", target, "->", baseline_prep.name)
